# Exercise 11 — Buffett's Alpha

Frazzini, Kabiller & Pedersen (2018), *Buffett's Alpha*, **Financial Analysts Journal**
74(4), 35–55.

The seminar answers the paper. This notebook answers the part of the paper that needs
data: it rebuilds the alpha ladder on Berkshire Hathaway's own listed returns, splits
the sample at the end of the published study, and then asks the Lecture 11 question —
whether either result has enough statistical power to mean anything.

**Data.** Berkshire Hathaway Class A monthly total returns, May 1985 to December 2025,
derived from S&P Capital IQ adjusted closing prices. The Fama–French five factors and
the momentum factor from the [Kenneth R. French Data
Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), and
the betting-against-beta factor from [AQR Capital
Management](https://www.aqr.com/Insights/Datasets) — the same files as Exercise 10.

**Two departures from the paper, both deliberate.**

1. The paper runs October 1976 to March 2017. Capital IQ's price history for BRK.A
   begins in **April 1985**, so we lose Berkshire's strongest decade. Every level in
   this notebook is therefore lower than the published one. The *pattern* is what
   replicates, not the magnitudes.
2. The paper's quality factor is AQR's QMJ. We use Fama–French **RMW** (robust minus
   weak profitability) instead. It is a narrower measure of quality — profitability
   only, where QMJ also carries growth, safety and payout — so it should and does load
   less strongly.

**How to work with this notebook.** Start with **Runtime → Restart and run all**, then
read top to bottom. Section headings match the question numbers on the exercise sheet.
Q3 and Q6 are answered from the paper alone and have no code here.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, save_results
setup_style()

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def as_percent(df, cols):
    # display copy with the named columns scaled to percentage points
    out = df.copy()
    out[cols] = out[cols] * 100
    return out.to_string(float_format=lambda v: f"{v:8.2f}")

### The data

Three files, three formats. The Berkshire returns are ours and arrive clean. The French
files are committed verbatim, copyright lines and all, so the cleaning happens here in
front of you — the same `read_french` helper as Exercise 10. The AQR workbook has
nineteen rows of disclaimer above the header and one column per country.

Everything is monthly and in decimals. The merge is an inner join, so the sample is
whatever all four files have in common.

In [ ]:
BASE11 = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
          "Exercise_11/data/")
BASE10 = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
          "Exercise_10/data/")


def read_french(url, skiprows):
    # read one block of a Kenneth French CSV into monthly decimal returns
    df = pd.read_csv(url, skiprows=skiprows)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = df["date"].astype(str).str.strip()
    is_month = df["date"].str.fullmatch(r"\d{6}")
    stop = (~is_month).values.argmax() if (~is_month).any() else len(df)
    df = df.iloc[:stop]
    df = df.set_index(pd.PeriodIndex(df["date"], freq="M")).drop(columns="date")
    df.columns = [c.strip() for c in df.columns]
    return df.astype(float).replace([-99.99, -999.0], np.nan) / 100


brk = pd.read_csv(BASE11 + "brk_monthly_returns.csv", parse_dates=["Date"])
brk = brk.set_index(pd.PeriodIndex(brk["Date"], freq="M"))[["BRK"]]

ff5 = read_french(BASE10 + "F-F_Research_Data_5_Factors_2x3.csv", 4)
mom = read_french(BASE10 + "F-F_Momentum_Factor.csv", 13)

bab = pd.read_excel(BASE10 + "Betting_Against_Beta_Equity_Factors_Monthly.xlsx",
                    sheet_name="BAB Factors", skiprows=18)
bab["date"] = pd.to_datetime(bab["DATE"], format="%m/%d/%Y")
bab = bab.set_index(pd.PeriodIndex(bab["date"], freq="M"))[["USA"]]
bab = bab.rename(columns={"USA": "BAB"}).dropna()

data = brk.join([ff5, mom, bab], how="inner").dropna()
data["rx"] = data["BRK"] - data["RF"]          # Berkshire excess return

FACTORS = ["Mkt-RF", "SMB", "HML", "Mom", "BAB", "RMW"]

# the two sub-periods: the published sample, and everything after it
A = data.loc[:"2017-12"]
B = data.loc["2018-01":]

print(f"merged sample {data.index[0]} to {data.index[-1]}, "
      f"T = {len(data)} months ({len(data)/12:.1f} years)")
print(f"  period A     {A.index[0]} to {A.index[-1]}, T = {len(A)}")
print(f"  period B     {B.index[0]} to {B.index[-1]}, T = {len(B)}")

## Q1 — The track record

Before any model: how good was it, and against what?

The four numbers that matter are the average excess return, the volatility, the ratio of
the two, and the same ratio for the market. The paper reports a Sharpe ratio of 0.79 for
Berkshire against 0.49 for the market, and calls the first "very good but not
superhuman". Reproduce those four numbers on each sub-period and say what changed.

In [ ]:
def summary(df, label):
    rx, mkt = df["rx"], df["Mkt-RF"]
    return pd.Series({
        "months": len(df),
        "BRK mean": rx.mean() * 12,
        "BRK vol": rx.std() * np.sqrt(12),
        "BRK Sharpe": rx.mean() / rx.std() * np.sqrt(12),
        "MKT mean": mkt.mean() * 12,
        "MKT vol": mkt.std() * np.sqrt(12),
        "MKT Sharpe": mkt.mean() / mkt.std() * np.sqrt(12),
    }, name=label)


track = pd.DataFrame([summary(data, "full 1985-2025"),
                      summary(A, "A  1985-2017"),
                      summary(B, "B  2018-2025")])

print("means and volatilities in % p.a.\n")
print(as_percent(track, ["BRK mean", "BRK vol", "MKT mean", "MKT vol"]))

In [ ]:
fig1, ax = plt.subplots(figsize=(7.5, 4.5))
for series, name, colour in [(data["BRK"], "Berkshire Hathaway", FHNW["blue"]),
                             (data["Mkt-RF"] + data["RF"], "US market", FHNW["navy"])]:
    ax.plot(data.index.to_timestamp(), (1 + series).cumprod(),
            label=name, color=colour, lw=1.5)
ax.axvline(pd.Timestamp("2018-01-01"), color=FHNW["red"], lw=1.0, ls="--")
ax.annotate("published\nsample ends", xy=(pd.Timestamp("2018-01-01"), 3),
            xytext=(-8, 0), textcoords="offset points",
            color=FHNW["red"], fontsize=9, ha="right", va="center")
ax.set_yscale("log")
ax.set_ylabel("value of 1 USD invested, log scale")
ax.set_xlabel("")
ax.legend(loc="upper left")
plt.show()

<!-- solution -->
### What Q1 shows

Over the whole sample Berkshire earned **13.9 % p.a.** in excess of the T-bill against
the market's **9.2 %**, on **21.2 %** volatility against **15.5 %**. That is a Sharpe
ratio of **0.66** against **0.59** — a real edge, and a much smaller one than the
reputation.

The split is where it gets interesting. Through 2017 the ratio is **0.68** against
**0.56**. From 2018 it is **0.57** against **0.71**: on the last eight years, an index
fund had the better risk-adjusted return.

Two things students get wrong. The first is comparing our 0.66 with the paper's 0.79 and
concluding one of them is an error. Both are correct on their own samples; we simply
start nine years later and lose the decade in which Berkshire compounded fastest. The
second is reading the log-scale chart as "the outperformance stopped in 2018". Look
again. On a log scale a constant gap means equal growth rates, and the gap stops widening
much around 2000 — eighteen years before our break. What the split at 2018 captures is
not where Berkshire changed but where the *published study* ends, which is a different
thing and the honest way to label it.

## Q2 — Leverage

The paper estimates Berkshire's leverage at about 1.7 to 1 at market values, from

$$L_t = \frac{TA_t - Cash_t}{Equity_t}$$

and then makes a one-line argument: apply 1.7× to the market and you get roughly 12.7 %,
far short of Berkshire's 18.6 %, so leverage cannot be the whole story.

Compute the same ratio on book values, and then re-run the paper's one-line argument on
each of our two sub-periods.

In [ ]:
bs = pd.read_csv(BASE11 + "brk_balance_sheet_annual.csv",
                 parse_dates=["FiscalYearEnd"])
bs["L"] = (bs["TotalAssets"] - bs["Cash"]) / bs["TotalEquity"]

print(f"book leverage, {bs['FiscalYearEnd'].dt.year.min()}"
      f"-{bs['FiscalYearEnd'].dt.year.max()}:  "
      f"mean {bs['L'].mean():.2f}   median {bs['L'].median():.2f}   "
      f"min {bs['L'].min():.2f}   max {bs['L'].max():.2f}")

L_PAPER = 1.7
print("\nthe paper's one-line argument, re-run:\n")
for df, label in [(data, "full 1985-2025"), (A, "A  1985-2017"), (B, "B  2018-2025")]:
    lev_mkt = L_PAPER * df["Mkt-RF"].mean() * 12
    print(f"  {label:16s}  1.7 x market = {lev_mkt:6.1%}   "
          f"Berkshire = {df['rx'].mean() * 12:6.1%}   "
          f"{'market wins' if lev_mkt > df['rx'].mean() * 12 else 'Berkshire wins'}")

<!-- solution -->
### What Q2 shows

Book leverage averages **1.90** over 1992–2025, against the paper's 1.7 at market
values. The two are close enough to confirm the order of magnitude and different enough
to be worth a sentence: book equity understates Berkshire's market equity for most of
this period, which pushes the book ratio up.

The paper's one-line argument does not survive the update. On the published sample it
worked: 1.7 × 7.5 % = 12.7 % against 18.6 %, a wide gap. On ours it is **14.4 % against
14.6 %** through 2017 — a tie — and **20.6 % against 11.0 %** afterwards, where the
leveraged index wins outright.

The right conclusion is not that the paper was wrong. It is that this particular
argument was never strong. It compares a leveraged market portfolio with Berkshire while
ignoring that the two have different betas, different volatilities and different
drawdowns; a 1.7× market position had 26 % volatility over our sample against
Berkshire's 21 %. The comparison is a rhetorical device, and it happened to point the
right way on one sample. The regressions below are the argument that carries weight.

The cash line deserves one caution. Our `Cash` column is cash and equivalents only, and
Berkshire holds very large short-term Treasury positions outside it. A wider cash
definition lowers the numerator and moves leverage materially. This is exactly the kind
of definitional choice Lecture 11 warned about under "test assets are not a technicality".

## Q4 — The alpha ladder

This is Table 4 of the paper. Regress Berkshire's excess return on a widening set of
factors and watch the intercept:

$$r_t - r^f_t = \alpha + \beta_1 MKT_t + \beta_2 SMB_t + \beta_3 HML_t
+ \beta_4 UMD_t + \beta_5 BAB_t + \beta_6 RMW_t + \varepsilon_t$$

Five rungs: the market alone, then the three Fama–French factors, then momentum, then
betting-against-beta, then profitability. Run the ladder separately on each sub-period.

The question is not whether the alpha falls — adding regressors can only reduce the
unexplained part. The question is *which* rung takes it, and whether what remains is
still distinguishable from zero.

In [ ]:
LADDER = [("CAPM", ["Mkt-RF"]),
          ("+ FF3", ["Mkt-RF", "SMB", "HML"]),
          ("+ UMD", ["Mkt-RF", "SMB", "HML", "Mom"]),
          ("+ BAB", ["Mkt-RF", "SMB", "HML", "Mom", "BAB"]),
          ("+ RMW", ["Mkt-RF", "SMB", "HML", "Mom", "BAB", "RMW"])]


def ladder(df, label):
    rows = {}
    for name, cols in LADDER:
        res = sm.OLS(df["rx"], sm.add_constant(df[cols])).fit()
        row = {"alpha": res.params.iloc[0] * 12, "t(alpha)": res.tvalues.iloc[0]}
        row.update({c: res.params[c] for c in cols})
        row["R2"] = res.rsquared
        rows[name] = row
    out = pd.DataFrame(rows).T[["alpha", "t(alpha)"] + FACTORS + ["R2"]]
    print(f"\n{label}   (alpha in % p.a.)\n")
    print(as_percent(out, ["alpha"]))
    return out


lad_A = ladder(A, "Period A — May 1985 to December 2017")
lad_B = ladder(B, "Period B — January 2018 to December 2025")

In [ ]:
fig2, ax = plt.subplots(figsize=(7.5, 4.5))
x = np.arange(len(LADDER))
width = 0.38
for offset, tab, name, colour in [(-width / 2, lad_A, "A  1985-2017", FHNW["blue"]),
                                  (+width / 2, lad_B, "B  2018-2025", FHNW["orange"])]:
    bars = ax.bar(x + offset, tab["alpha"], width, label=name, color=colour)
    for rect, t in zip(bars, tab["t(alpha)"]):
        ax.annotate(f"t={t:.2f}", (rect.get_x() + rect.get_width() / 2,
                                   rect.get_height()),
                    textcoords="offset points",
                    xytext=(0, 3 if rect.get_height() >= 0 else -12),
                    ha="center", fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, [n for n, _ in LADDER])
ax.set_ylim(-0.022, 0.105)
ax.set_ylabel("alpha, % p.a.")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.legend()
plt.show()

<!-- solution -->
### What Q4 shows

**Period A reproduces the paper.** The alpha starts at **9.09 % p.a. (t = 2.66)** and
falls to **4.26 % (t = 1.28)** — it loses statistical significance at the BAB rung and
never recovers it. The loadings tell the story the paper tells: **SMB −0.42** (large
caps), **HML +0.34** (cheap), **BAB +0.17** (safe, low beta), **RMW +0.27**
(profitable), **UMD −0.01** (no trend-chasing whatsoever). Beta is **0.65**. Buffett
bought large, cheap, safe, profitable companies and levered them, and once you price
those four things his alpha is no longer distinguishable from zero.

The magnitudes are roughly half the paper's, because our sample starts nine years later.
The *shape* is identical, and that is the replication.

**Period B has nothing to explain.** The alpha is **2.08 % (t = 0.39)** at the CAPM rung
and **−0.56 % (t = −0.12)** at the end. It is already gone before any style factor
enters. Note also that R² jumps from **0.21** to **0.43** in the market model alone:
Berkshire has become far more market-like. Beta rises from 0.65 to 0.74.

The mistake to head off: reading the falling alpha as evidence that Buffett had no skill.
Everything in the ladder was published *decades after* he started doing it — BAB in 2014,
QMJ in 2018. Explaining a return with factors identified in hindsight says where the
return came from, not that earning it was easy. The paper is explicit about this and the
seminar should be too.

## Q5 — Does either result have enough power?

Lecture 11 gave the rule

$$t(\alpha) \;\approx\; IR \cdot \sqrt{T}$$

with $T$ in years. Check it against both sub-periods, then use it in the other
direction: how long a track record would each information ratio need before its
t-statistic reached 2?

Then the question that decides how Period B should be read. **Suppose Berkshire's
Period-A skill continued into Period B, completely unchanged.** How often would eight
years of data detect it?

In [ ]:
rows = {}
for df, label in [(A, "A  1985-2017"), (B, "B  2018-2025")]:
    res = sm.OLS(df["rx"], sm.add_constant(df["Mkt-RF"])).fit()
    ir = res.params.iloc[0] / res.resid.std()           # monthly, so scale-free
    years = len(df) / 12
    rows[label] = {"IR (ann.)": ir * np.sqrt(12),
                   "T (years)": years,
                   "IR*sqrt(T)": ir * np.sqrt(12) * np.sqrt(years),
                   "actual t(alpha)": res.tvalues.iloc[0],
                   "years for t=2": (2 / (ir * np.sqrt(12))) ** 2}
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f"{v:10.2f}"))

In [ ]:
# how often would eight years detect Period A's skill, if it were still there?
res_A = sm.OLS(A["rx"], sm.add_constant(A["Mkt-RF"])).fit()
alpha_m, resid_sd = res_A.params.iloc[0], res_A.resid.std()
n_months, n_sims = len(B), 20_000

draws = rng.normal(alpha_m, resid_sd, size=(n_sims, n_months))
t_sim = draws.mean(1) / (draws.std(1, ddof=1) / np.sqrt(n_months))

t_B = sm.OLS(B["rx"], sm.add_constant(B["Mkt-RF"])).fit().tvalues.iloc[0]
print(f"true alpha {alpha_m * 12:.2%} p.a., residual vol {resid_sd * np.sqrt(12):.1%} p.a., "
      f"{n_months} months, {n_sims:,} draws")
print(f"  share reaching t > 1.96      {(t_sim > 1.96).mean():6.1%}")
print(f"  median t-statistic           {np.median(t_sim):6.2f}")
print(f"  share with a NEGATIVE alpha  {(draws.mean(1) < 0).mean():6.1%}")
print(f"  actually observed in B       {t_B:6.2f}")

In [ ]:
fig3, ax = plt.subplots(figsize=(7.5, 4.5))
counts, edges, _ = ax.hist(t_sim, bins=60, color=FHNW["blue"], alpha=0.55,
                           edgecolor="none")
mids = (edges[:-1] + edges[1:]) / 2
ax.bar(mids[mids > 1.96], counts[mids > 1.96], width=np.diff(edges)[0],
       color=FHNW["green"], label=f"detected: {(t_sim > 1.96).mean():.0%}")
ax.axvline(1.96, color=FHNW["navy"], lw=1.4, ls="--", label="t = 1.96")
ax.axvline(t_B, color=FHNW["red"], lw=1.8,
           label=f"observed in period B, t = {t_B:.2f}")
ax.set_ylim(0, counts.max() * 1.35)
ax.set_xlabel("t-statistic of the CAPM alpha over an eight-year window")
ax.set_ylabel("simulated track records")
ax.legend(loc="upper right")
plt.show()

<!-- solution -->
### What Q5 shows

**The rule works, twice.** Period A: an information ratio of **0.47** over **32.7 years**
predicts **2.70**; the regression gives **2.66**. Period B: **0.14** over **8.0 years**
predicts **0.40**; the regression gives **0.39**. Two decimal places, on real data, from
a formula you can do in your head.

Run it backwards and the numbers become uncomfortable. At Period A's information ratio a
manager needs **17.9 years** to reach t = 2. At Period B's, **200 years**. Almost no
track record anyone will ever show you is long enough to establish what its owner claims
it establishes.

**And this is why Period B settles nothing.** If Berkshire's Period-A skill had continued
into Period B completely unchanged, eight years of data would have produced a significant
alpha only **27 %** of the time. The median t-statistic would have been **1.34**. And
**8.8 %** of the time the estimated alpha would have come out *negative* — on a manager
whose true alpha is 9 % a year.

So the honest reading of Period B is not "the alpha is gone". It is: *eight years cannot
tell the difference between an alpha of zero and an alpha of nine percent, and we should
stop pretending otherwise.* That is the joint-hypothesis lesson of Lecture 11 arriving on
the most famous track record in finance — and it cuts against the conclusion most people
would like to draw from the same numbers.

### Where this leaves the seminar

Q3 and Q6 have no code. The 13F filing and the insurance float are read off the paper,
and both matter for the same reason: they are the parts of Berkshire that a factor
regression cannot see. Float at 1.72 % — roughly three points below the T-bill rate — is
not a style tilt. It is a funding advantage nobody else in the sample had, and no
long–short factor portfolio can replicate it.

The exercise sheet closes with the investment-committee question. Take the two ladders
and the power table into it: you now know what an eight-year track record can and cannot
establish, and you have seen the most celebrated record in the industry fail to clear the
bar on its most recent eight years.